# Deep Agents 第 1 阶段：harness 基础

对应 `codes/deepagents/index.md` 第 1 阶段，讲义见同目录 `README.md`。

先运行下一格加载 `DEEPSEEK_API_KEY`（可放 `codes/langgraph/local.env`）。有密钥时走真实模型；没有时用脚本化假模型跑通同样的工具路径，方便离线复现。答案已补全。


In [1]:
import os
from pathlib import Path
from typing import ClassVar

for p in [Path.cwd(), *Path.cwd().parents]:
    env = p / "codes/langgraph/local.env"
    if env.is_file():
        for line in env.read_text().splitlines():
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
        break

from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

HAS_KEY = bool(os.environ.get("DEEPSEEK_API_KEY"))

class ScriptedModel(GenericFakeChatModel):
    """Scripted fake model that records bind_tools names."""

    seen_tools: ClassVar[list] = []

    def bind_tools(self, tools, **kw):
        names = sorted(t.name if hasattr(t, "name") else t["name"] for t in tools)
        type(self).seen_tools.append(names)
        return self

def call(name, args, i):
    return AIMessage("", tool_calls=[{"name": name, "args": args, "id": f"c{i}"}])

if HAS_KEY:
    from langchain.chat_models import init_chat_model
    model = init_chat_model(f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}")
    print("live model ->", type(model).__name__)
else:
    model = None
    print("no DEEPSEEK_API_KEY -> scripted offline answers")


no DEEPSEEK_API_KEY -> scripted offline answers


## 1. 最小 deep agent 与它的工具集

用 `create_deep_agent(model=model)` 建一个代理，让它「把三条 Python 并发要点写进 /notes.md」。

1. 打印 `out.keys()` 和 `out["files"]`。
2. 文件在磁盘上吗？为什么？
3. 工具列表里有没有 `write_todos`？（提示：看 0.7 发布说明）


In [2]:
from deepagents import create_deep_agent

# 作答
if HAS_KEY:
    agent = create_deep_agent(
        model=model,
        system_prompt="把用户要的要点写进 /notes.md，写完简短确认即可。",
    )
    out = agent.invoke({
        "messages": [{"role": "user", "content": "把三条 Python 并发要点写进 /notes.md"}],
    })
    tool_names = sorted({
        tc["name"]
        for m in out["messages"]
        if getattr(m, "tool_calls", None)
        for tc in m.tool_calls
    })
else:
    ScriptedModel.seen_tools.clear()
    notes = (
        "1. asyncio: cooperative concurrency on one thread, good for IO\n"
        "2. threading: OS threads, GIL-limited, good for blocking IO\n"
        "3. multiprocessing: processes bypass GIL, good for CPU work\n"
    )
    script = [
        call("write_file", {"file_path": "/notes.md", "content": notes}, 1),
        AIMessage("wrote /notes.md"),
    ]
    sm = ScriptedModel(messages=iter(script))
    agent = create_deep_agent(model=sm)
    out = agent.invoke({"messages": [{"role": "user", "content": "把三条 Python 并发要点写进 /notes.md"}]})
    tool_names = ScriptedModel.seen_tools[-1]

print("out.keys() ->", sorted(out.keys()))
files_view = {
    k: (v.get("content", v) if isinstance(v, dict) else v)
    for k, v in (out.get("files") or {}).items()
}
print("out['files'] ->", files_view)
print("disk? -> no. Default StateBackend keeps files in state['files'], not on disk.")
print("write_todos in tools? ->", "write_todos" in tool_names, "(opt-in since 0.7 via TodoListMiddleware)")
print("tools sample ->", tool_names)


out.keys() -> ['files', 'messages']
out['files'] -> {'/notes.md': '1. asyncio: cooperative concurrency on one thread, good for IO\n2. threading: OS threads, GIL-limited, good for blocking IO\n3. multiprocessing: processes bypass GIL, good for CPU work\n'}
disk? -> no. Default StateBackend keeps files in state['files'], not on disk.
write_todos in tools? -> False (opt-in since 0.7 via TodoListMiddleware)
tools sample -> ['delete', 'edit_file', 'glob', 'grep', 'ls', 'read_file', 'task', 'write_file']


## 2. 打开计划工具

给第 1 题的代理加上 `TodoListMiddleware`，换一个多步任务（例如「对比 asyncio / threading / multiprocessing，分三节写进 /report.md」）。

1. 用 `agent.stream(..., stream_mode="updates")` 观察 `todos` 的状态变化。
2. 和不加时比，模型调用次数和最终文件有什么差别？


In [3]:
from langchain.agents.middleware import TodoListMiddleware

# 作答
prompt = "对比 asyncio / threading / multiprocessing，分三节写进 /report.md"

if HAS_KEY:
    agent = create_deep_agent(
        model=model,
        middleware=[TodoListMiddleware()],
        system_prompt="先用 write_todos 列出三节计划并更新状态，再把正文写进 /report.md。",
    )
    print("--- stream updates (todos / files) ---")
    for chunk in agent.stream({"messages": [{"role": "user", "content": prompt}]}, stream_mode="updates"):
        for node, update in chunk.items():
            if isinstance(update, dict) and "todos" in update:
                print(f"node={node} todos ->", update["todos"])
            if isinstance(update, dict) and "files" in update:
                print(f"node={node} files keys ->", list(update["files"]))
    out = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
else:
    ScriptedModel.seen_tools.clear()
    script = [
        call("write_todos", {"todos": [
            {"content": "asyncio section", "status": "in_progress"},
            {"content": "threading section", "status": "pending"},
            {"content": "multiprocessing section", "status": "pending"},
        ]}, 1),
        call("write_todos", {"todos": [
            {"content": "asyncio section", "status": "completed"},
            {"content": "threading section", "status": "in_progress"},
            {"content": "multiprocessing section", "status": "pending"},
        ]}, 2),
        call("write_file", {
            "file_path": "/report.md",
            "content": "# asyncio\n# threading\n# multiprocessing\n",
        }, 3),
        call("write_todos", {"todos": [
            {"content": "asyncio section", "status": "completed"},
            {"content": "threading section", "status": "completed"},
            {"content": "multiprocessing section", "status": "completed"},
        ]}, 4),
        AIMessage("report ready"),
    ]
    sm = ScriptedModel(messages=iter(script))
    agent = create_deep_agent(model=sm, middleware=[TodoListMiddleware()])
    print("--- stream updates (todos) ---")
    for chunk in agent.stream({"messages": [{"role": "user", "content": prompt}]}, stream_mode="updates"):
        for node, update in chunk.items():
            if isinstance(update, dict) and "todos" in update:
                print(f"node={node} todos ->", update["todos"])
    sm2 = ScriptedModel(messages=iter([
        call("write_todos", {"todos": [
            {"content": "asyncio section", "status": "completed"},
            {"content": "threading section", "status": "completed"},
            {"content": "multiprocessing section", "status": "completed"},
        ]}, 1),
        call("write_file", {
            "file_path": "/report.md",
            "content": "# asyncio\n# threading\n# multiprocessing\n",
        }, 2),
        AIMessage("report ready"),
    ]))
    out = create_deep_agent(model=sm2, middleware=[TodoListMiddleware()]).invoke(
        {"messages": [{"role": "user", "content": prompt}]}
    )

print("final todos ->", out.get("todos"))
print("final /report.md ->", (out.get("files") or {}).get("/report.md"))
print("diff vs no-todo -> tools gain write_todos; state gains todos channel;")
print("model usually plans then writes, so more rounds; file still lands in /report.md.")
print("write_todos is a no-op tool: value is the attention anchor each turn.")


--- stream updates (todos) ---
node=tools todos -> [{'content': 'asyncio section', 'status': 'in_progress'}, {'content': 'threading section', 'status': 'pending'}, {'content': 'multiprocessing section', 'status': 'pending'}]
node=tools todos -> [{'content': 'asyncio section', 'status': 'completed'}, {'content': 'threading section', 'status': 'in_progress'}, {'content': 'multiprocessing section', 'status': 'pending'}]
node=tools todos -> [{'content': 'asyncio section', 'status': 'completed'}, {'content': 'threading section', 'status': 'completed'}, {'content': 'multiprocessing section', 'status': 'completed'}]
final todos -> [{'content': 'asyncio section', 'status': 'completed'}, {'content': 'threading section', 'status': 'completed'}, {'content': 'multiprocessing section', 'status': 'completed'}]
final /report.md -> {'content': '# asyncio\n# threading\n# multiprocessing\n', 'encoding': 'utf-8', 'created_at': '2026-10-30T13:56:38.146941+00:00', 'modified_at': '2026-10-30T13:56:38.146941

## 3. 子代理与上下文隔离

定义一个 `critic` 子代理（`name` / `description` / `system_prompt`），主代理先写草稿，再派 `critic` 审阅。

1. 在 `out["messages"]` 里找到 `task` 对应的 `ToolMessage`，它包含子代理的哪部分输出？
2. 子代理的 `system_prompt` 出现在父代理消息里了吗？
3. 把 `description` 改成一个含糊的词，再跑一次，主代理还会派活吗？


In [4]:
# 作答
CRITIC_PROMPT = "You are a strict reviewer. Return exactly three concrete edit notes."

if HAS_KEY:
    critic = {
        "name": "critic",
        "description": "Review the draft in /draft.md and return actionable edit notes",
        "system_prompt": CRITIC_PROMPT,
    }
    agent = create_deep_agent(
        model=model,
        subagents=[critic],
        system_prompt="Write a draft to /draft.md, then task critic to review it.",
    )
    out = agent.invoke({
        "messages": [{"role": "user", "content": "Write a short GIL draft to /draft.md and ask critic to review."}],
    })
else:
    sub_model = ScriptedModel(messages=iter([
        AIMessage("1) Define GIL up front\n2) Add an asyncio contrast\n3) Drop colloquial phrasing"),
    ]))
    script = [
        call("write_file", {
            "file_path": "/draft.md",
            "content": "GIL allows only one thread to run Python bytecode at a time.",
        }, 1),
        call("task", {"description": "review /draft.md", "subagent_type": "critic"}, 2),
        AIMessage("revised from review notes"),
    ]
    sm = ScriptedModel(messages=iter(script))
    agent = create_deep_agent(
        model=sm,
        subagents=[{
            "name": "critic",
            "description": "Review the draft in /draft.md and return actionable edit notes",
            "system_prompt": CRITIC_PROMPT,
            "model": sub_model,
        }],
    )
    out = agent.invoke({"messages": [{"role": "user", "content": "draft + critic review"}]})

task_msgs = [
    m for m in out["messages"]
    if getattr(m, "type", None) == "tool" and getattr(m, "name", "") == "task"
]
if not task_msgs:
    task_msgs = [m for m in out["messages"] if getattr(m, "type", None) == "tool"]
print("task ToolMessage content ->", getattr(task_msgs[0], "content", None) if task_msgs else None)
print("Q1 -> only the subagent final message, not intermediate tool turns.")
print(
    "Q2 -> subagent system_prompt does NOT appear in parent messages. seen? ->",
    any(CRITIC_PROMPT in str(getattr(m, "content", "")) for m in out["messages"]),
)

if HAS_KEY:
    vague = {"name": "critic", "description": "help", "system_prompt": CRITIC_PROMPT}
    agent_v = create_deep_agent(model=model, subagents=[vague], system_prompt="Delegate when useful.")
    out_v = agent_v.invoke({"messages": [{"role": "user", "content": "Review: GIL is annoying."}]})
    tasked = any(
        getattr(m, "tool_calls", None) and any(tc.get("name") == "task" for tc in m.tool_calls)
        for m in out_v["messages"]
    )
    print("Q3 vague description -> routing gets unreliable; task seen this run ->", tasked)
else:
    print("Q3 -> description is the routing signal; a vague 'help' usually stops stable critic dispatch.")


task ToolMessage content -> 1) Define GIL up front
2) Add an asyncio contrast
3) Drop colloquial phrasing
Q1 -> only the subagent final message, not intermediate tool turns.
Q2 -> subagent system_prompt does NOT appear in parent messages. seen? -> False
Q3 -> description is the routing signal; a vague 'help' usually stops stable critic dispatch.


## 4. 换 backend：长期记忆

用 `CompositeBackend` 把 `/memories/` 路由到 `StoreBackend`（`InMemoryStore`），其余仍走 `StateBackend`。

1. 第一个 `thread_id` 里让代理把「用户偏好：回答简洁」写进 `/memories/pref.md`。
2. 换一个新 `thread_id`，让代理先读这个文件再回答。哪些文件跨线程还在？

注意：用 `checkpointer=MemorySaver()` 才有 thread 概念。


In [5]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

# 作答
store = InMemoryStore()
checkpointer = MemorySaver()
backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=lambda rt: ("demo-user",), store=store)},
)

if HAS_KEY:
    agent = create_deep_agent(
        model=model,
        backend=backend,
        store=store,
        checkpointer=checkpointer,
        system_prompt="Long-term prefs go under /memories/; drafts elsewhere.",
    )
    out1 = agent.invoke(
        {"messages": [{"role": "user", "content": "Write 'user preference: be concise' to /memories/pref.md, and a scratch line to /scratch.md"}]},
        config={"configurable": {"thread_id": "t1"}},
    )
    out2 = agent.invoke(
        {"messages": [{"role": "user", "content": "Read /memories/pref.md first, then introduce asyncio in one sentence."}]},
        config={"configurable": {"thread_id": "t2"}},
    )
else:
    sm1 = ScriptedModel(messages=iter([
        call("write_file", {"file_path": "/memories/pref.md", "content": "user preference: be concise"}, 1),
        call("write_file", {"file_path": "/scratch.md", "content": "tmp draft"}, 2),
        AIMessage("done"),
    ]))
    agent = create_deep_agent(model=sm1, backend=backend, store=store, checkpointer=checkpointer)
    out1 = agent.invoke(
        {"messages": [{"role": "user", "content": "write pref + scratch"}]},
        config={"configurable": {"thread_id": "t1"}},
    )
    sm2 = ScriptedModel(messages=iter([
        call("read_file", {"file_path": "/memories/pref.md"}, 1),
        AIMessage("asyncio: one thread runs concurrent IO via an event loop."),
    ]))
    agent2 = create_deep_agent(model=sm2, backend=backend, store=store, checkpointer=checkpointer)
    out2 = agent2.invoke(
        {"messages": [{"role": "user", "content": "prefer concise; introduce asyncio"}]},
        config={"configurable": {"thread_id": "t2"}},
    )

print("thread t1 files ->", list((out1.get("files") or {}).keys()))
print("thread t2 files ->", list((out2.get("files") or {}).keys()))
print("store search (/memories/) ->", store.search(("demo-user",)))
print("conclusion -> /scratch.md stays on StateBackend (per thread); /memories/pref.md is in StoreBackend across threads.")
print("t2 final reply ->", out2["messages"][-1].content if out2.get("messages") else None)


thread t1 files -> ['/scratch.md']
thread t2 files -> []
store search (/memories/) -> [Item(namespace=['demo-user'], key='/pref.md', value={'content': 'user preference: be concise', 'encoding': 'utf-8', 'created_at': '2026-10-30T13:56:38.271470+00:00', 'modified_at': '2026-10-30T13:56:38.271470+00:00'}, created_at='2026-10-30T13:56:38.271498+00:00', updated_at='2026-10-30T13:56:38.271499+00:00', score=None)]
conclusion -> /scratch.md stays on StateBackend (per thread); /memories/pref.md is in StoreBackend across threads.
t2 final reply -> asyncio: one thread runs concurrent IO via an event loop.


## 5. 对比题（不写代码）

用自己的话回答：

1. `create_deep_agent` 和第 6 阶段的预置 ReAct agent，在图结构上是不是同一种东西？差别在哪一层？
2. 一个「固定三步」的报表流程，和一个「调研一个陌生领域」的任务，各该选哪个？
3. 为什么说 `write_todos` 是 no-op 工具却依然可能有用？


In [6]:
# 作答
answers = """1. Same family of graph: both are LangGraph CompiledStateGraph model<->tool loops.
   Difference is the harness/middleware layer: create_deep_agent = create_agent plus
   Filesystem/SubAgent/Summarization/PatchToolCalls and swappable backends.
   Stage-6 prebuilt ReAct is the thinner loop.

2. Fixed 3-step report -> hand-written LangGraph (or plain create_agent).
   Open-ended research -> create_deep_agent (unknown step count, file store, subagents, optional todos).

3. write_todos has no business side effect; it only writes into state.todos so every
   turn can see the plan. A no-op tool buys an attention anchor against goal drift.
"""
print(answers)


1. Same family of graph: both are LangGraph CompiledStateGraph model<->tool loops.
   Difference is the harness/middleware layer: create_deep_agent = create_agent plus
   Filesystem/SubAgent/Summarization/PatchToolCalls and swappable backends.
   Stage-6 prebuilt ReAct is the thinner loop.

2. Fixed 3-step report -> hand-written LangGraph (or plain create_agent).
   Open-ended research -> create_deep_agent (unknown step count, file store, subagents, optional todos).

3. write_todos has no business side effect; it only writes into state.todos so every
   turn can see the plan. A no-op tool buys an attention anchor against goal drift.

